# ARC Prize 2026 — ARC-AGI-3 "Cognitive USB" Autonomous Agent

### Unified Single-Notebook Architecture powered by HBLLM Core

Instead of monolithic, custom-coded agent plumbing, **ARC-AGI-3 is treated as a Cognitive USB peripheral (a virtual game console)** plugged directly into the HBLLM Autonomic Brain:

1. **`ArcAgiGameDriver` (`BaseDriver`)**: Exposes the 2D grid display as sensory input (`DriverInput`) and maps motor commands (`UP`, `DOWN`, `LEFT`, `RIGHT`, `INTERACT`, `CLICK`) through standard `DriverAction`.
2. **`EpistemicCognitiveBrain`**: Embedded mental simulation and curiosity engine performing online motor grounding, A* mental planning, and barrier learning.
3. **Canonical Autonomic Loop**: `driver_manager.execute_cognitive_step()` drives the full Observe $\rightarrow$ Simulate $\rightarrow$ Act $\rightarrow$ Learn cycle in a single, clean step.

In [ ]:
# ==============================================================================
# 1. RUNTIME INITIALIZATION & ENVIRONMENT SETUP
# ==============================================================================
from __future__ import annotations

import glob
import json
import logging
import os
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np

# Path resolution for local repo and Kaggle environment
for p in [
    Path("."),
    Path(".."),
    Path("core"),
    Path("/kaggle/working"),
    Path("/kaggle/input/hbllm-core"),
]:
    if p.exists() and str(p.resolve()) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

# Optional Kaggle wheel installer (if running in offline Kaggle sandbox)
wheel_dirs = set()
for p in glob.glob("/kaggle/input/**/*.whl", recursive=True):
    wheel_dirs.add(os.path.dirname(p))
if wheel_dirs:
    for wd in sorted(wheel_dirs):
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-index", f"--find-links={wd}", "arc-agi"],
            check=False,
        )

# Check for official Arcade environment
try:
    from arc_agi import Arcade

    HAS_OFFICIAL_ARCADE = True
    print("✅ Official 'arc_agi' Arcade environment detected!")
except ImportError:
    HAS_OFFICIAL_ARCADE = False
    print(
        "ℹ️ Official 'arc_agi' not installed — falling back to embedded Synthetic Arcade simulator."
    )

# Check for HBLLM Core Biological Brain Faculties
try:
    from hbllm.hcir.world.autonomous_epistemic_engine import AutonomousEpistemicEngine

    HAS_CORE_ENGINE = True
    print("🧠 HBLLM Core Epistemic Engine (Biological Brain Faculties) detected!")
except ImportError:
    HAS_CORE_ENGINE = False
    print("ℹ️ HBLLM Core Engine not found on path — using embedded heuristic engine.")

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("arc_cognitive_usb")

In [ ]:
# ==============================================================================
# 2. COGNITIVE USB PROTOCOL & ARC-AGI-3 PERIPHERAL DRIVER
# ==============================================================================
from hbllm.drivers.base import (
    BaseDriver,
    DriverAction,
    DriverCapability,
    DriverFeedback,
    DriverInput,
    DriverModality,
    DriverStreamType,
    SynapticDeviceDescriptor,
)
from hbllm.drivers.manager import DriverManager


def _extract_grid(raw: Any) -> np.ndarray | None:
    """Safely extract 2D integer grid from bare arrays, temporal lists, or 3D buffers."""
    if raw is None:
        return None
    if isinstance(raw, (list, tuple)) and len(raw) > 0:
        raw = raw[-1]
    if isinstance(raw, np.ndarray):
        if raw.ndim == 3 and len(raw) > 0:
            raw = raw[-1]
        return np.asarray(raw, dtype=int)
    return None


class ArcAgiGameDriver(BaseDriver):
    """Cognitive USB Driver wrapping ARC-AGI-3 interactive grid environments."""

    def __init__(self, name: str = "arc_console", env: Any = None) -> None:
        desc = SynapticDeviceDescriptor(
            device_id=name,
            vendor_id="arc_prize",
            device_type="bidirectional",
            modalities=[DriverModality.GRID_2D, DriverModality.STRUCTURED],
            stream_type=DriverStreamType.POLL,
            action_schema=[
                {"name": "MOVE_UP", "action_id": 1},
                {"name": "MOVE_DOWN", "action_id": 2},
                {"name": "MOVE_LEFT", "action_id": 3},
                {"name": "MOVE_RIGHT", "action_id": 4},
                {"name": "INTERACT", "action_id": 5},
                {"name": "CLICK_CELL", "action_id": 6, "parameters": {"x": "int", "y": "int"}},
            ],
        )
        super().__init__(
            name=name,
            capabilities={
                DriverCapability.DISCRETE_ACTIONS,
                DriverCapability.STEP_BASED_EXECUTION,
                DriverCapability.SPATIAL_2D,
            },
            descriptor=desc,
        )
        self.env = env
        self.current_frame = None

    def connect(self, target: Any) -> bool:
        self.env = target
        self.is_connected = True
        return True

    def disconnect(self) -> None:
        self.is_connected = False
        self.env = None

    def get_action_list(self, inputs: DriverInput) -> list[DriverAction]:
        avail = inputs.metadata.get("available_actions", [1, 2, 3, 4, 5, 6])
        schema_map = {item["action_id"]: item for item in self.descriptor.action_schema}
        actions = []
        for aid in avail:
            item = schema_map.get(aid, {})
            name = item.get("name", str(aid))
            params = item.get("parameters", {})
            actions.append(DriverAction(action_id=aid, semantic_intent=name, parameters=params))
        return actions

    def set_frame(self, frame: Any) -> None:
        self.current_frame = frame

    def get_inputs(self) -> DriverInput:
        frame_obj = self.current_frame
        raw_grid = getattr(frame_obj, "frame", None)
        if raw_grid is None and isinstance(frame_obj, dict):
            raw_grid = frame_obj.get("frame") or frame_obj.get("grid")

        grid = _extract_grid(raw_grid)

        avail = getattr(frame_obj, "available_actions", None)
        if avail is None and isinstance(frame_obj, dict):
            avail = frame_obj.get("available_actions", [1, 2, 3, 4, 5, 6])
        avail = list(avail) if avail is not None else [1, 2, 3, 4, 5, 6]

        return DriverInput(
            raw_data=grid,
            source_id=self.name,
            modality=DriverModality.GRID_2D,
            metadata={
                "frame_obj": frame_obj,
                "available_actions": avail,
                "levels_completed": getattr(frame_obj, "levels_completed", 0),
                "win_levels": getattr(frame_obj, "win_levels", 1),
                "state": getattr(
                    getattr(frame_obj, "state", None), "name", str(getattr(frame_obj, "state", ""))
                ),
            },
        )

    def send_output(self, action: DriverAction) -> Any:
        act_id = action.action_id
        params = action.parameters or {}
        data = params if params else None
        next_frame = self.env.step(act_id, data=data)
        self.current_frame = next_frame
        return next_frame

    def process_feedback(self, raw_result: Any) -> DriverFeedback:
        if raw_result is None:
            return DriverFeedback(success=False, terminated=True)

        state_str = getattr(
            getattr(raw_result, "state", None), "name", str(getattr(raw_result, "state", ""))
        )

        lvl = getattr(raw_result, "levels_completed", 0)
        win_levels = getattr(raw_result, "win_levels", 1)

        is_win = state_str == "WIN" or lvl >= win_levels
        is_done = is_win or state_str == "GAME_OVER"
        reward = 1.0 if is_win else (0.5 if lvl > 0 else 0.0)

        grid = _extract_grid(getattr(raw_result, "frame", None))
        avail = getattr(raw_result, "available_actions", [1, 2, 3, 4, 5, 6])
        return DriverFeedback(
            success=True,
            reward=reward,
            terminated=is_done,
            causal_delta=grid,
            info={
                "levels_completed": lvl,
                "state": state_str,
                "win_levels": win_levels,
                "available_actions": list(avail) if avail is not None else [1, 2, 3, 4, 5, 6],
            },
        )

In [ ]:
# ==============================================================================
# 3. EPISTEMIC COGNITIVE BRAIN (BIOLOGICAL BRAIN FACULTIES)
# ==============================================================================
class EpistemicCognitiveBrain:
    """Embedded cognitive engine operating inside HBLLM DriverManager.

    Powered directly by HBLLM Core's AutonomousEpistemicEngine:
    1. Saccadic Visual Attention & Foveal Saliency Filtering.
    2. Spatiotemporal Hippocampal Projection (x, y, t) & Periodic Hazard Avoidance.
    3. Compound Intuitive Physics & 2x2/Tunnel Deadlock Pruning.
    4. Prefrontal Working Memory & Hierarchical Subgoal Schemas.
    5. Episodic Long-Term Memory (Cognitive Memory) Hydration.
    """

    def __init__(self) -> None:
        self.core_engine = AutonomousEpistemicEngine() if HAS_CORE_ENGINE else None
        self.prev_grid: np.ndarray | None = None
        self.last_action: int | None = None
        self.current_game_id: str | None = None

    def reset_episode(self, retain_dynamics: bool = True) -> None:
        if self.core_engine is not None:
            self.core_engine.reset_episode(retain_dynamics=retain_dynamics)
        self.prev_grid = None
        self.last_action = None

    def load_cognitive_memory(self, game_id: str) -> None:
        """Autonomous Epistemic Grounding: Let the biological brain discover world dynamics cleanly in real time."""
        base_gid = game_id.split("-")[0].strip().lower()
        self.current_game_id = base_gid

    def observe(
        self, driver_input: DriverInput, perception_data: dict[str, Any] | None = None
    ) -> None:
        grid = _extract_grid(driver_input.raw_data)
        if grid is None or not isinstance(grid, np.ndarray):
            return
        self.prev_grid = grid.copy()

    def decide(self, available_actions: list[DriverAction], source_id: str = "") -> DriverAction:
        if not available_actions:
            return DriverAction(action_id=1)

        avail_map = {a.action_id: a for a in available_actions}
        avail_ids = list(avail_map.keys())

        # 1. Primary: HBLLM Core Biological Brain Decisions
        if self.core_engine is not None and self.prev_grid is not None:
            action_schemas = [
                {
                    "action_id": a.action_id,
                    "name": a.semantic_intent,
                    "parameters": a.parameters,
                }
                for a in available_actions
            ]
            act_id, action_data = self.core_engine.decide(
                self.prev_grid, avail_ids, action_schemas=action_schemas
            )
            params = action_data if action_data else {}
            self.last_action = act_id
            return DriverAction(action_id=act_id, parameters=params)

        # 2. Fallback Heuristic
        return available_actions[0]

    def update(self, action: DriverAction, feedback: DriverFeedback, source_id: str = "") -> None:
        if self.core_engine is not None and feedback.causal_delta is not None:
            curr_grid = _extract_grid(feedback.causal_delta)
            avail = feedback.info.get("available_actions", [1, 2, 3, 4, 5, 6])
            st = feedback.info.get("state", "")
            lvl = int(feedback.info.get("levels_completed", 0))
            is_win = st == "WIN" or lvl > getattr(self, "_last_lvl", 0)
            is_lost = st == "GAME_OVER"
            if curr_grid is not None:
                self.core_engine.assimilate_feedback(
                    curr_grid,
                    avail,
                    is_win=is_win,
                    is_lost=is_lost,
                    action=action.action_id,
                    action_data=action.parameters,
                )
            self._last_lvl = lvl

In [ ]:
# ==============================================================================
# 4. SYNTHETIC PROCEDURAL ARCADE SIMULATOR (OFFLINE VERIFICATION)
# ==============================================================================
class SyntheticArcadeGame:
    """Procedural ARC-AGI-3 Maze Game for self-contained testing."""

    def __init__(self, size: int = 8, win_levels: int = 2) -> None:
        self.size = size
        self.levels_completed = 0
        self.win_levels = win_levels
        self.state = "RUNNING"
        self.available_actions = [1, 2, 3, 4]
        self._init_level()

    def _init_level(self) -> None:
        self.grid = np.zeros((self.size, self.size), dtype=int)
        self.grid[3, : self.size - 2] = 5  # Barrier wall with narrow doorway
        self.avatar_pos = (1, 1)
        self.goal_pos = (self.size - 2, 1)
        self.grid[self.avatar_pos] = 2  # Red Avatar
        self.grid[self.goal_pos] = 3  # Green Goal Target
        self.step_count = 0

    @property
    def frame(self) -> np.ndarray:
        return self.grid.copy()

    def reset(self) -> Any:
        self._init_level()
        self.state = "RUNNING"
        return self

    def step(self, action_id: int, data: dict[str, Any] | None = None) -> Any:
        if action_id == 0:
            self._init_level()
            self.state = "RUNNING"
            return self
        self.step_count += 1
        deltas = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}
        if action_id in deltas:
            dr, dc = deltas[action_id]
            nr, nc = self.avatar_pos[0] + dr, self.avatar_pos[1] + dc
            if 0 <= nr < self.size and 0 <= nc < self.size and self.grid[nr, nc] != 5:
                self.grid[self.avatar_pos] = 0
                self.avatar_pos = (nr, nc)
                self.grid[self.avatar_pos] = 2

        if self.avatar_pos == self.goal_pos:
            self.levels_completed += 1
            if self.levels_completed >= self.win_levels:
                self.state = "WIN"
            else:
                self._init_level()

        if self.step_count > 120:
            self.state = "GAME_OVER"

        return self

In [ ]:
# ==============================================================================
# 5. UNIFIED EVALUATION & COMPETITION HARNESS (ALL 25 GAMES x ALL 183 LEVELS)
# ==============================================================================
def evaluate_game(
    env: Any, game_name: str, max_steps: int = 150
) -> tuple[int, int, str, float, int]:
    """Run a complete game using the Cognitive USB DriverManager pipeline."""
    driver = ArcAgiGameDriver(name="arc_console", env=env)
    manager = DriverManager()
    brain = EpistemicCognitiveBrain()
    brain.load_cognitive_memory(game_name)
    manager.set_cognitive_engine(brain)

    manager.register(driver)
    manager.bind("arc_console", target=env)

    initial_frame = env.reset()
    driver.set_frame(initial_frame)
    brain.reset_episode(retain_dynamics=True)
    brain.load_cognitive_memory(game_name)

    target_levels = int(getattr(initial_frame, "win_levels", 1) or 1)
    start_time = time.time()
    completed_prev = 0
    game_steps = 0

    final_state = "RUNNING"
    # Canonical Autonomic Loop with In-Budget Level Retries
    for step in range(1, max_steps + 1):
        feedback = manager.execute_cognitive_step()
        game_steps += 1
        lvl = int(feedback.info.get("levels_completed", 0))
        state_str = feedback.info.get("state", "NOT_FINISHED")
        final_state = state_str

        if lvl > completed_prev:
            completed_prev = lvl
            brain.reset_episode(retain_dynamics=True)

        if lvl >= target_levels or state_str == "WIN":
            final_state = "WIN"
            break

        # Level failure / death retry within the allowed step budget
        if feedback.terminated or state_str == "GAME_OVER":
            brain.reset_episode(retain_dynamics=True)
            try:
                reset_frame = env.step(0)
            except Exception:
                try:
                    reset_frame = env.reset()
                except Exception:
                    break
            reset_state = getattr(
                getattr(reset_frame, "state", None), "name", str(getattr(reset_frame, "state", ""))
            )
            if reset_state == "GAME_OVER":
                break
            driver.set_frame(reset_frame)

    elapsed = time.time() - start_time
    return completed_prev, target_levels, final_state, elapsed, game_steps


def main_benchmark():
    print("=" * 75, flush=True)
    print(
        "🚀 ARC-AGI-3 COGNITIVE USB BENCHMARK EVALUATION (ALL 25 GAMES x ALL 183 LEVELS)",
        flush=True,
    )
    print("   Powered by HBLLM Core Biological Brain Faculties", flush=True)
    print("=" * 75, flush=True)

    scorecard = {}
    total_passed = 0
    total_possible = 0
    total_steps = 0
    start_all = time.time()

    if HAS_OFFICIAL_ARCADE:
        arcade = Arcade()
        environments = arcade.get_environments()
        print(
            f"Evaluating against all {len(environments)} official Arcade games across all levels...",
            flush=True,
        )
        for idx, env_meta in enumerate(environments, 1):
            gid = env_meta.game_id
            title = getattr(env_meta, "title", gid)
            baseline = getattr(env_meta, "baseline_actions", None) or [100]

            print(f"[{idx:2d}/{len(environments)}] Game: {title} ({gid})", flush=True)
            print(
                f"    Baseline Actions: {baseline} | Tags: {getattr(env_meta, 'tags', [])}",
                flush=True,
            )

            env = arcade.make(gid)
            if env is None:
                continue

            target_levels = int(len(baseline) or 1)
            level_budget = [int(b * 2.5) if b > 0 else 150 for b in baseline]
            if len(level_budget) < target_levels:
                level_budget.extend([150] * (target_levels - len(level_budget)))
            total_budget = sum(level_budget[:target_levels])

            passed, target, state, el, steps = evaluate_game(env, gid, max_steps=total_budget)
            scorecard[gid] = {
                "title": title,
                "passed": passed,
                "target": target,
                "state": state,
                "elapsed": round(el, 2),
                "steps": steps,
            }
            total_passed += passed
            total_possible += target
            total_steps += steps

            status_icon = "🏆 WIN" if passed >= target else ("⭐ PARTIAL" if passed > 0 else "❌ 0")
            print(
                f"    [{status_icon}] {passed}/{target} levels ({state}) in {el:.2f}s ({steps} steps)",
                flush=True,
            )
            print(
                f"    Cumulative: {total_passed}/{total_possible} levels ({(total_passed / max(1, total_possible)) * 100:.1f}%)",
                flush=True,
            )

            with open("all_25_games_scorecard.json", "w") as f:
                json.dump(scorecard, f, indent=2)
    else:
        print("Evaluating against procedural synthetic arcade test environments...", flush=True)
        for idx, size in enumerate([6, 8, 10], 1):
            env = SyntheticArcadeGame(size=size, win_levels=2)
            gid = f"synth_{size}x{size}"
            passed, target, state, el, steps = evaluate_game(env, gid)
            scorecard[gid] = f"{passed}/{target}"
            total_passed += passed
            total_possible += target
            status_icon = "✅ WIN" if passed >= target else "❌ INCOMPLETE"
            print(
                f"  [{idx}/3] {gid:<12}: {passed}/{target} levels ({status_icon}) in {el:.3f}s ({steps} steps)",
                flush=True,
            )

    total_el = time.time() - start_all
    pct = (total_passed / max(1, total_possible)) * 100.0
    print("=" * 75, flush=True)
    print(
        f"🏁 FINAL SCORECARD: {total_passed}/{total_possible} Total Levels Passed ({pct:.1f}%)",
        flush=True,
    )
    print(
        f"   Total Steps: {total_steps} | Total Time: {total_el:.1f}s ({total_el / 60.0:.1f} min)",
        flush=True,
    )
    print("=" * 75, flush=True)


# Run evaluation
main_benchmark()